# quant-retrieval, rebuilt from scratch on a free GPU

Nothing to upload, no Drive folder to create, no setup. Open it, set the runtime
to a T4, and Run all.

It clones the repo, rebuilds the 26,152 document corpus from the public Stack
Exchange dump, trains every model, evaluates every pipeline, and hands you a zip
of the results at the end.

## Why it retrains rather than downloading checkpoints

The earlier version asked for a checkpoint to be uploaded, because that model was
trained on a laptop GPU and retraining it here gives slightly different weights.
That was friction for no real gain. Retraining everything on one machine is
better: every number in the repo then comes from the same hardware and the whole
thing reproduces from a clean clone, which is worth more than matching numbers
that were themselves produced on a machine nobody else has.

Expect the committed numbers to shift a little as a result. Same data, same
seeds, same code, different floating point.

## Time

About 50 minutes on a T4. The order is deliberate: the questions worth answering
come first, so if the session dies late you already have what matters. Each stage
prints as it finishes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader \
  || print("NO GPU. Runtime > Change runtime type > T4 GPU, then Run all again.")

In [ ]:
%cd /content
!rm -rf quant-retrieval
!git clone -q https://github.com/melihgiray/quant-retrieval.git
%cd /content/quant-retrieval

# Colab ships torch built for this GPU. Installing our pinned version would
# replace it with a slower or broken build, so install the package without its
# dependencies and add only what Colab lacks.
!pip install -q py7zr faiss-cpu
!pip install -q -e . --no-deps

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## The run

Stages in order of what they answer. Everything is a call into `scripts/`, so
the cloud and a laptop run identical code.

In [ ]:
STAGES = [
    ("data", [
        "python scripts/download_data.py",
        "python scripts/build_dataset.py",
    ]),
    ("baselines", [
        "python scripts/evaluate.py --config configs/bm25.yaml",
        "python scripts/evaluate.py --config configs/minilm_frozen.yaml",
    ]),
    ("train the retriever", [
        "python scripts/train.py --config configs/base.yaml",
        "python scripts/evaluate.py --config configs/minilm_tuned_epoch1.yaml",
        "python scripts/evaluate.py --config configs/minilm_tuned_epoch2.yaml",
        "python scripts/evaluate.py --config configs/minilm_tuned_epoch3.yaml",
        "python scripts/evaluate.py --config configs/hybrid.yaml",
    ]),
    ("negatives and the reranker", [
        "python scripts/mine_negatives.py --config configs/negatives.yaml",
        "python scripts/train_reranker.py --config configs/reranker_mixed.yaml",
        # The decisive number. A working reranker beats four random documents
        # well above 90 percent of the time. The mined-only one managed 43.
        "python scripts/probe_reranker.py --checkpoint checkpoints/reranker_mixed/epoch-2",
        "python scripts/evaluate.py --config configs/bm25_rerank_mixed.yaml",
        "python scripts/evaluate.py --config configs/dense_rerank_mixed.yaml",
        "python scripts/evaluate.py --config configs/hybrid_rerank_mixed.yaml",
    ]),
    ("significance", [
        "python scripts/compare_runs.py --baseline results/minilm_frozen_val.json"
        " --candidate results/minilm_tuned_epoch3_val.json --metric ndcg_at_10",
        "python scripts/compare_runs.py --baseline results/bm25_val.json"
        " --candidate results/minilm_frozen_val.json --metric ndcg_at_10",
        "python scripts/compare_runs.py --baseline results/minilm_tuned_epoch3_val.json"
        " --candidate results/hybrid_val.json --metric ndcg_at_10",
        "python scripts/compare_runs.py --baseline results/minilm_tuned_epoch3_val.json"
        " --candidate results/hybrid_val.json --metric recall_at_100",
    ]),
    ("latency and the index", [
        "pytest -q tests/test_ann.py",
        "python scripts/export_index.py --checkpoint checkpoints/minilm_tuned/epoch-3",
        "python scripts/profile_pipeline.py --config configs/hybrid.yaml --queries 100",
        "python scripts/profile_pipeline.py --config configs/minilm_tuned_epoch3.yaml --queries 100",
    ]),
    ("ablations, the least urgent", [
        "python scripts/train.py --config configs/minilm_batch16.yaml",
        "python scripts/evaluate.py --config configs/minilm_batch16_epoch3.yaml",
        "python scripts/train.py --config configs/minilm_batch32.yaml",
        "python scripts/evaluate.py --config configs/minilm_batch32_epoch3.yaml",
        "python scripts/train.py --config configs/minilm_batch128.yaml",
        "python scripts/evaluate.py --config configs/minilm_batch128_epoch3.yaml",
        "python scripts/train.py --config configs/minilm_cls.yaml",
        "python scripts/evaluate.py --config configs/minilm_cls_epoch3.yaml",
        "python scripts/train.py --config configs/minilm_hardneg.yaml",
        "python scripts/evaluate.py --config configs/minilm_hardneg_epoch3.yaml",
        "python scripts/train_reranker.py --config configs/reranker.yaml",
        "python scripts/evaluate.py --config configs/bm25_rerank.yaml",
        "python scripts/evaluate.py --config configs/dense_rerank.yaml",
        "python scripts/evaluate.py --config configs/hybrid_rerank.yaml",
        "python scripts/compare_runs.py --baseline results/minilm_tuned_epoch3_val.json"
        " --candidate results/minilm_cls_epoch3_val.json --metric ndcg_at_10",
        "python scripts/compare_runs.py --baseline results/minilm_tuned_epoch3_val.json"
        " --candidate results/minilm_hardneg_epoch3_val.json --metric ndcg_at_10",
        "python scripts/compare_runs.py --baseline results/minilm_tuned_epoch3_val.json"
        " --candidate results/minilm_hardneg_epoch3_val.json --metric recall_at_100",
        "python scripts/compare_runs.py --baseline results/minilm_batch32_epoch3_val.json"
        " --candidate results/minilm_tuned_epoch3_val.json --metric ndcg_at_10",
    ]),
]

import subprocess, sys, time
started_all = time.time()
for name, commands in STAGES:
    print("\n" + "#" * 70)
    print(f"# STAGE: {name}   [{(time.time() - started_all) / 60:.0f} min elapsed]")
    print("#" * 70, flush=True)
    for command in commands:
        print("\n>>>", command, flush=True)
        started = time.time()
        completed = subprocess.run(command, shell=True)
        print(f"<<< exit {completed.returncode} after {time.time() - started:.0f}s", flush=True)
        if completed.returncode != 0:
            print("\nSTOPPED. Everything finished so far is still in results/ "
                  "and the next cell will package it.", flush=True)
            raise SystemExit(1)
print(f"\nALL DONE in {(time.time() - started_all) / 60:.0f} minutes")

## Take the results

In [ ]:
# The decisive number first.
import json, pathlib
probe = pathlib.Path("results/reranker_mixed_probe.json")
if probe.exists():
    for entry in json.loads(probe.read_text())["splits"]:
        verdict = "WORKS" if entry["top_one_accuracy"] > 0.9 else "STILL BROKEN"
        print(f"{entry['split']:>5}: {entry['top_one_accuracy']:.1%} correct against four "
              f"random documents (chance {entry['chance']:.0%})   {verdict}")
    print("\nThe reranker trained on mined negatives alone scored 43% and 28%.")
else:
    print("no probe output, the reranker stage did not finish")

In [ ]:
!zip -qr /content/results.zip results
from google.colab import files
files.download("/content/results.zip")
print("Downloaded results.zip. Unzip it over the repo folder, replacing results/.")